# DATA PRE-PROCESSING in GSE69914 - Advanced study of epigenetic mechanisms in the development of neoplasms 

In this notebook, I will evaluate and define the most appropriate **pre-processing operations**.

### Libraries

In [1]:
!pip install methylprep==1.7.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 14.6 MB/s eta 0:00:0000:010:01


In [2]:
import methylprep; print(f"✅ methylprep {methylprep.__version__} available")


✅ methylprep 1.7.0 available


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import median_abs_deviation
import warnings
from scipy.stats import mannwhitneyu
from scipy.stats import gaussian_kde
from __future__ import annotations
import os, re, math
from typing import Iterable, Optional, Set
from polars import selectors as cs
import polars as pl

%config HistoryManager.enabled = False
warnings.filterwarnings('ignore')


## 1. Import and Data Structure

The processed dataset is imported from the LZ4-compressed **`.parquet`** file generated in the previous step. The structure is already optimized for analysis.

* **File format:** Columnar **`Parquet`** (LZ4 compression) for fast I/O.
* **Rows:** Samples (one per tissue).
* **Columns:**
    * **`id_tissue`**: Unique sample identifier.
    * **`label`**: Numeric class code (**`Int8`**).
    * **`cg`, `ch`**: Methylation probes (**`Float32`**).


In [4]:
# IMPORT DATA STRUCTURE
PARQUET_PATH = "/kaggle/input/gse69914-parquet/GSE69914.parquet"  # CHANGE HERE!! #
ID_COL = "id_tissue"
LABEL_COL = "label"

# Import dataset
GSE69914 = pl.scan_parquet(PARQUET_PATH)
print("✅ Dataset successfully loaded.")


✅ Dataset successfully loaded.


## 2. Data Validation and Integrity Check
* **Dimensions:** The dataset contains **407 samples × 485,514 CpG loci**, matching the expected layout (samples × features).
* **Data types:** Confirmed schema — `id_tissue: String`, `label: Int8`, `probes: Float32`, ensuring compact storage and numerical precision.
* **β-value range:** All values within `[0.000000, 0.997110]`, correctly bounded in [0, 1].
* **Missing values:** No NaN entries detected across any probe — overall missing rate **0%**.

*The methylation matrix is structurally sound, numerically consistent, and fully complete — no cleaning or imputation required before further analysis.*


In [5]:
# DATA VALIDATION CHECK
# Dimensions (rows × columns)
n_rows = GSE69914.select(pl.len().alias("rows")).collect(streaming=True)["rows"][0]
n_cols = len(GSE69914.columns)
print(f"- Dimensions: ({n_rows:,}, {n_cols:,}) → samples × CpG loci")

# Data types
schema = GSE69914.schema
id_dtype = schema.get("id_tissue", None)
label_dtype = schema.get("label", None)

# Detect a few probe columns (cg/ch)
probe_cols = [c for c in schema if c.startswith(("cg", "ch"))][:10]
probe_dtypes = {c: schema[c] for c in probe_cols}

print(f"- Data types:")
print(f"  id_tissue: {id_dtype}")
print(f"  label: {label_dtype}")
print(f"  probes (sample): {list(probe_dtypes.values())[:3]} ...")
print("  ✅ Expected types: (Int8, Float32)")

# Value range check (β-values ∈ [0, 1])
# Compute min and max across all probe columns (efficient aggregation)
probe_exprs = [pl.col(c) for c in schema if c.startswith(("cg", "ch"))]
beta_range = GSE69914.select([
    pl.min_horizontal(probe_exprs).alias("beta_min"),
    pl.max_horizontal(probe_exprs).alias("beta_max")
]).collect(streaming=True)

beta_min = float(beta_range["beta_min"][0])
beta_max = float(beta_range["beta_max"][0])

print(f"- β-value range: [{beta_min:.6f}, {beta_max:.6f}]")
if 0 <= beta_min <= 1 and 0 <= beta_max <= 1:
    print("  ✅ All β-values within expected [0, 1] range.")
else:
    print("  ⚠️  Warning: values outside expected range detected.")


- Dimensions: (407, 485,514) → samples × CpG loci
- Data types:
  id_tissue: String
  label: Int8
  probes (sample): [Float32, Float32, Float32] ...
  ✅ Expected types: (Int8, Float32)
- β-value range: [0.000000, 0.997110]
  ✅ All β-values within expected [0, 1] range.


In [6]:
# Select only numeric CpG/CH probe columns (exclude id_tissue, label)
probe_cols = [c for c in GSE69914.columns if c.startswith(("cg", "ch"))]

# Global NaN count across all probes
null_total = (
    GSE69914.select([pl.col(c).null_count().alias(c) for c in probe_cols])
    .select(pl.sum_horizontal(pl.all()).alias("total_null"))
    .collect(streaming=True)["total_null"][0]
)

# Total number of values (for percentage)
total_values = len(probe_cols) * GSE69914.select(pl.len().alias("rows")).collect(streaming=True)["rows"][0]
missing_rate = null_total / total_values * 100

# Sample check for columns and rows with NaN (to confirm all zero)
col_nulls = (
    GSE69914.select([pl.col(c).null_count().alias(c) for c in probe_cols[:10]])
    .collect(streaming=True)
    .to_dict(as_series=False)
)

print(f"- Total NaN count across all probes: {null_total:,}")
print(f"- Overall missing rate: {missing_rate:.6f}%")

if null_total == 0:
    print("  ✅ No missing entries detected. The methylation matrix is complete.")
else:
    print("  ⚠️ Missing values detected — consider filtering or imputation.")


- Total NaN count across all probes: 0
- Overall missing rate: 0.000000%
  ✅ No missing entries detected. The methylation matrix is complete.


## 3. Technical Filtering

In [17]:
# TECHNICAL FILTERS ON POLARS BETA-MATRIX
NAEEM_CSV   = "/kaggle/input/filtering-cpg-naeem/filtering_table.csv"            # Naeem's list 2014
PIDSLEY_CSV = "/kaggle/input/filtering-cpg-pidsley/Chen2013_full_blacklist.csv"  # Pidsley's list 2016
ZHOU_TSV    = "/kaggle/input/filtering-cpg-zhou/EPIC.anno.GRCh38.tsv"            # Zhou annotation with MASK_* columns 2016

def load_naeem_drop_set(path: str,
                        probe_cands: Iterable[str]=("probe","IlmnID","ID_REF","cg","Name"),
                        flag_cands:  Iterable[str]=("Flag(discard/keep)","flag(discard/keep)","flag","flag_discard_keep")) -> Set[str]:
    """Return probes flagged as 'discard' in Naeem table."""
    df = pl.read_csv(path)
    probe_col = next((c for c in probe_cands if c in df.columns), None)
    flag_col  = next((c for c in df.columns if str(c).lower().strip().startswith("flag")), None) or next((c for c in flag_cands if c in df.columns), None)
    assert probe_col and flag_col, f"Naeem file columns not found. Seen: {df.columns}"
    probes = df[probe_col].cast(pl.Utf8).str.strip_chars()
    flags  = df[flag_col].cast(pl.Utf8).str.to_lowercase()
    return set(probes.filter(flags.str.contains("discard")).to_list())

def load_pidsley_drop_set(path: str, skip_first_lines: int = 1) -> Set[str]:
    """
    Robust loader for Pidsley2016 blacklist (CSV/TSV):
    - Try comma and tab, pick the parse with more columns.
    - Extract any CpG-like ids (cg* or ch*), regardless of column name.
    """
    df_comma = pl.read_csv(path, has_header=True, infer_schema_length=10000,
                           separator=",", skip_rows=skip_first_lines)
    try:
        df_tab = pl.read_csv(path, has_header=True, infer_schema_length=10000,
                             separator="\t", skip_rows=skip_first_lines)
    except Exception:
        df_tab = None
    cand = df_tab if (df_tab is not None and df_tab.width > df_comma.width) else df_comma

    cpg_like_cols = [c for c in cand.columns
                     if ("cg" in str(c).lower()) or ("ilmn" in str(c).lower())
                     or ("probe" in str(c).lower()) or ("id" in str(c).lower())]
    if not cpg_like_cols:
        cpg_like_cols = [cand.columns[0]]

    series_list = cand.select(pl.concat_list([pl.col(c).cast(pl.Utf8) for c in cpg_like_cols]).alias("_u")).to_series()
    s = series_list.explode().cast(pl.Utf8).str.strip_chars()
    values = s.to_list()
    return set(x for x in values if isinstance(x, str) and (x.startswith("cg") or x.startswith("ch")))

def zhou_mask_ids(tsv_path: str,
                  id_col_candidates: Iterable[str] = ("probeID","IlmnID","Name")) -> Set[str]:
    """
    Read only ID + MASK_* columns from Zhou TSV (or TSV.GZ), coerce MASK_* to bool,
    and return the set of probes to drop (any MASK_* == True).
    """
    # discover header
    hdr = pl.read_csv(tsv_path, separator="\t", has_header=True, n_rows=1, ignore_errors=True)
    id_col = next((c for c in id_col_candidates if c in hdr.columns), None)
    assert id_col, f"ID column not found in Zhou TSV. Seen: {hdr.columns}"
    mask_cols = [c for c in hdr.columns if str(c).upper().startswith("MASK")]
    assert mask_cols, "No MASK_* columns found in Zhou TSV."

    # read only needed columns
    df = pl.read_csv(tsv_path, separator="\t", has_header=True, columns=[id_col]+mask_cols,
                     dtypes={id_col: pl.Utf8}, ignore_errors=True)

    # normalize MASK_* to boolean
    truthy = {"true","1","t","yes","y"}
    falsy  = {"false","0","f","no","n",""}
    def mask_to_bool(colname: str) -> pl.Expr:
        s = pl.col(colname).cast(pl.Utf8).str.to_lowercase()
        return (pl.when(s.is_in(list(truthy))).then(True)
                  .when(s.is_in(list(falsy))).then(False)
                  .otherwise(False)
                  .alias(colname))
    df_norm = df.select([pl.col(id_col).cast(pl.Utf8).str.strip_chars().alias(id_col)] + [mask_to_bool(c) for c in mask_cols])

    # any MASK_* == True -> drop
    mask_any = pl.any_horizontal([pl.col(c) for c in mask_cols])
    bad = df_norm.filter(mask_any)[id_col].to_list()
    return set(bad)

# Accept both DataFrame and LazyFrame; detect layout # ELE TI HO MESSO QUESTA OPZIONE NEL CASO VOLESSI USARE PANDAS DOVREBBE FUNZIONARE
lf = GSE69914.lazy() if isinstance(GSE69914, pl.DataFrame) else GSE69914  # ensure LazyFrame -> IL MIO
schema_cols = list(lf.schema.keys())
n_cols_before = len(schema_cols)
n_rows_before = lf.select(pl.len()).collect().item()

id_col_candidates = ("IlmnID","probeID","ID_REF","CpG","cg_id","Name")
id_col = next((c for c in id_col_candidates if c in schema_cols), None)

if id_col:
    # CpGs as rows: collect only the ID column (unique)
    present_cpgs = set(
        lf.select(pl.col(id_col).cast(pl.Utf8).str.strip_chars().alias("_id")).unique()
          .collect().get_column("_id").to_list()
    )
    layout = "rows"
else:
    # CpGs as columns (wide matrix): include cg* and ch*
    cpg_cols = [c for c in schema_cols if isinstance(c, str) and (c.startswith("cg") or c.startswith("ch"))]
    present_cpgs = set(cpg_cols)
    layout = "cols"

# Load lists and intersect with present CpGs 
print("Loading technical filter lists...")
S_naeem_all = load_naeem_drop_set(NAEEM_CSV)
S_pidsley_all  = load_pidsley_drop_set(PIDSLEY_CSV)
S_zhou_all  = zhou_mask_ids(ZHOU_TSV)

S_naeem = S_naeem_all & present_cpgs
S_pidsley  = S_pidsley_all  & present_cpgs
S_zhou  = S_zhou_all  & present_cpgs

print(f"• Naeem 2014 (discard flag)  → in dataset: {len(S_naeem):,} / total in list: {len(S_naeem_all):,}")
print(f"• Pidsley 2016 blacklist     → in dataset: {len(S_pidsley):,} / total in list: {len(S_pidsley_all):,}")
print(f"• Zhou MASK_* 2016           → in dataset: {len(S_zhou):,} / total in list: {len(S_zhou_all):,}")

removed_cpgs = S_naeem | S_pidsley | S_zhou

# -------- 3) Apply filter lazily + save removed list --------
if layout == "rows":
    lf_filt = lf.filter(~pl.col(id_col).is_in(list(removed_cpgs)))
else:
    keep_cols = [c for c in schema_cols if c not in removed_cpgs]
    lf_filt = lf.select(keep_cols)

pl.DataFrame({"removed_cpg": sorted(removed_cpgs)}).write_csv("removed_cpgs.csv")

# Shapes after (lazy)
n_cols_after = len(lf_filt.schema)
n_rows_after = lf_filt.select(pl.len()).collect().item()

print("\nSummary:")
print(f"  • Samples × Probes (before): {n_rows_before} × {n_cols_before}")
print(f"  • Samples × Probes (after):  {n_rows_after} × {n_cols_after}")
print(f"  • Total removed CpGs:        {len(removed_cpgs):,}")
print(f"    - from Naeem: {len(S_naeem):,}   - from Pidsley: {len(S_pidsley):,}   - from Zhou: {len(S_zhou):,}")

# Materialize if needed:
GSE69914_filt = lf_filt  # .collect()  # uncomment to get a DataFrame

# -------- 4) Overlaps + detailed source log --------
N_only  = S_naeem - (S_pidsley | S_zhou)
C_only  = S_pidsley  - (S_naeem | S_zhou)
Z_only  = S_zhou  - (S_naeem | S_pidsley)
NC      = (S_naeem & S_pidsley) - S_zhou
NZ      = (S_naeem & S_zhou) - S_pidsley
CZ      = (S_pidsley  & S_zhou) - S_naeem
NCZ     = S_naeem & S_pidsley & S_zhou

print("\nOverlaps (present CpGs only):")
print(f"  • Naeem only:  {len(N_only):,}")
print(f"  • Pidsley only:   {len(C_only):,}")
print(f"  • Zhou only:   {len(Z_only):,}")
print(f"  • Naeem∩Pidsley:  {len(NC):,}")
print(f"  • Naeem∩Zhou:  {len(NZ):,}")
print(f"  • Pidsley∩Zhou:   {len(CZ):,}")
print(f"  • All three:   {len(NCZ):,}")

pl.DataFrame(
    [{"CpG_ID": cg,
      "From_Naeem": int(cg in S_naeem),
      "From_Pidsley":  int(cg in S_pidsley),
      "From_Zhou":  int(cg in S_zhou)} for cg in removed_cpgs]
).with_columns(
    (pl.col("From_Naeem")+pl.col("From_Pidsley")+pl.col("From_Zhou")).alias("Source_Count")
).sort(["Source_Count","CpG_ID"], descending=[True, False])\
 .write_csv("removed_cpgs_log.csv")

print("\nArtifacts:")
print("  • saved removed list → removed_cpgs.csv")
print("  • saved detailed log → removed_cpgs_log.csv")
# Optionally persist the filtered dataset:
# GSE69914_filt.collect().write_parquet("GSE69914_filtered.parquet")


Loading technical filter lists...
• Naeem 2014 (discard flag)  → in dataset: 190,672 / total in list: 190,672
• Pidsley 2016 blacklist     → in dataset: 39,737 / total in list: 67,030
• Zhou MASK_* 2016           → in dataset: 96,541 / total in list: 193,182

Summary:
  • Samples × Probes (before): 407 × 485514
  • Samples × Probes (after):  407 × 261378
  • Total removed CpGs:        224,136
    - from Naeem: 190,672   - from Pidsley: 39,737   - from Zhou: 96,541

Overlaps (present CpGs only):
  • Naeem only:  117,972
  • Pidsley only:   3,793
  • Zhou only:   21,852
  • Naeem∩Pidsley:  5,830
  • Naeem∩Zhou:  44,575
  • Pidsley∩Zhou:   7,819
  • All three:   22,295

Artifacts:
  • saved removed list → removed_cpgs.csv
  • saved detailed log → removed_cpgs_log.csv


In [5]:
# --- Technical filters on Polars beta-matrix (now with Chen-2013 cross-reactive) ---
import polars as pl
from typing import Iterable, Set, Dict, Tuple
from itertools import combinations

# Paths
NAEEM_CSV        = "/kaggle/input/filtering-cpg-naeem/filtering_table.csv"              # Naeem 2014
PIDSLEY_CSV      = "/kaggle/input/filtering-cpg-pidsley/Chen2013_full_blacklist.csv"    # Pidsley/Chen 2016/2013 (full blacklist)
CHEN_CROSS_CSV   = "/kaggle/input/filtering-cpg-chen-cross-reactive/chen_2013_cross_reactive.csv"  # Chen 2013 cross-reactive (1st col = CpG IDs)
ZHOU_TSV         = "/kaggle/input/filtering-cpg-zhou/EPIC.anno.GRCh38.tsv"              # Zhou 2016 MASK_* annotation

# ---------------- helpers to load lists ----------------
def load_naeem_drop_set(path: str,
                        probe_cands: Iterable[str]=("probe","IlmnID","ID_REF","cg","Name"),
                        flag_cands:  Iterable[str]=("Flag(discard/keep)","flag(discard/keep)","flag","flag_discard_keep")) -> Set[str]:
    """Return probes flagged as 'discard' in Naeem table."""
    df = pl.read_csv(path)
    probe_col = next((c for c in probe_cands if c in df.columns), None)
    flag_col  = next((c for c in df.columns if str(c).lower().strip().startswith("flag")), None) or next((c for c in flag_cands if c in df.columns), None)
    assert probe_col and flag_col, f"Naeem file columns not found. Seen: {df.columns}"
    probes = df[probe_col].cast(pl.Utf8).str.strip_chars()
    flags  = df[flag_col].cast(pl.Utf8).str.to_lowercase()
    return set(probes.filter(flags.str.contains("discard")).to_list())

def load_pidsley_drop_set(path: str, skip_first_lines: int = 1) -> Set[str]:
    """Robust loader for Pidsley/Chen full blacklist (CSV/TSV)."""
    df_comma = pl.read_csv(path, has_header=True, infer_schema_length=10000,
                           separator=",", skip_rows=skip_first_lines)
    try:
        df_tab = pl.read_csv(path, has_header=True, infer_schema_length=10000,
                             separator="\t", skip_rows=skip_first_lines)
    except Exception:
        df_tab = None
    cand = df_tab if (df_tab is not None and df_tab.width > df_comma.width) else df_comma

    cpg_like_cols = [c for c in cand.columns
                     if ("cg" in str(c).lower()) or ("ilmn" in str(c).lower())
                     or ("probe" in str(c).lower()) or ("id" in str(c).lower())]
    if not cpg_like_cols:
        cpg_like_cols = [cand.columns[0]]

    series_list = cand.select(pl.concat_list([pl.col(c).cast(pl.Utf8) for c in cpg_like_cols]).alias("_u")).to_series()
    s = series_list.explode().cast(pl.Utf8).str.strip_chars()
    values = s.to_list()
    return set(x for x in values if isinstance(x, str) and (x.startswith("cg") or x.startswith("ch")))

def load_chen_crossreactive_firstcol(path: str) -> Set[str]:
    """Chen 2013 cross-reactive list: CSV; keep ONLY the first column with probe IDs."""
    df = pl.read_csv(path, has_header=True, infer_schema_length=10000, separator=",")
    first_col = df.columns[0]
    s = df[first_col].cast(pl.Utf8).str.strip_chars()
    vals = s.to_list()
    return set(x for x in vals if isinstance(x, str) and (x.startswith("cg") or x.startswith("ch")))

def zhou_mask_ids(tsv_path: str,
                  id_col_candidates: Iterable[str] = ("probeID","IlmnID","Name")) -> Set[str]:
    """Read only ID + MASK_* columns from Zhou TSV/TSV.GZ and return probes to mask."""
    hdr = pl.read_csv(tsv_path, separator="\t", has_header=True, n_rows=1, ignore_errors=True)
    id_col = next((c for c in id_col_candidates if c in hdr.columns), None)
    assert id_col, f"ID column not found in Zhou TSV. Seen: {hdr.columns}"
    mask_cols = [c for c in hdr.columns if str(c).upper().startswith("MASK")]
    assert mask_cols, "No MASK_* columns found in Zhou TSV."

    df = pl.read_csv(tsv_path, separator="\t", has_header=True, columns=[id_col]+mask_cols,
                     dtypes={id_col: pl.Utf8}, ignore_errors=True)

    truthy = {"true","1","t","yes","y"}
    falsy  = {"false","0","f","no","n",""}
    def mask_to_bool(colname: str) -> pl.Expr:
        s = pl.col(colname).cast(pl.Utf8).str.to_lowercase()
        return (pl.when(s.is_in(list(truthy))).then(True)
                  .when(s.is_in(list(falsy))).then(False)
                  .otherwise(False)
                  .alias(colname))
    df_norm = df.select([pl.col(id_col).cast(pl.Utf8).str.strip_chars().alias(id_col)] + [mask_to_bool(c) for c in mask_cols])

    mask_any = pl.any_horizontal([pl.col(c) for c in mask_cols])
    bad = df_norm.filter(mask_any)[id_col].to_list()
    return set(bad)

# ---------------- dataset layout (LazyFrame/DataFrame) ----------------
lf = GSE69914.lazy() if isinstance(GSE69914, pl.DataFrame) else GSE69914
schema_cols = list(lf.schema.keys())
n_cols_before = len(schema_cols)
n_rows_before = lf.select(pl.len()).collect().item()

id_col_candidates = ("IlmnID","probeID","ID_REF","CpG","cg_id","Name")
id_col = next((c for c in id_col_candidates if c in schema_cols), None)

if id_col:
    present_cpgs = set(
        lf.select(pl.col(id_col).cast(pl.Utf8).str.strip_chars().alias("_id")).unique()
          .collect().get_column("_id").to_list()
    )
    layout = "rows"
else:
    cpg_cols = [c for c in schema_cols if isinstance(c, str) and (c.startswith("cg") or c.startswith("ch"))]
    present_cpgs = set(cpg_cols)
    layout = "cols"

# ---------------- load lists + intersect with present CpGs ----------------
print("Loading technical filter lists...")
S_naeem_all   = load_naeem_drop_set(NAEEM_CSV)
S_pidsley_all = load_pidsley_drop_set(PIDSLEY_CSV)
S_zhou_all    = zhou_mask_ids(ZHOU_TSV)
S_chenx_all   = load_chen_crossreactive_firstcol(CHEN_CROSS_CSV)  # NEW

S_naeem   = S_naeem_all   & present_cpgs
S_pidsley = S_pidsley_all & present_cpgs
S_zhou    = S_zhou_all    & present_cpgs
S_chenx   = S_chenx_all   & present_cpgs

print(f"• Naeem 2014 (discard flag)     → in dataset: {len(S_naeem):,}   / total in list: {len(S_naeem_all):,}")
print(f"• Pidsley/Chen (full blacklist) → in dataset: {len(S_pidsley):,} / total in list: {len(S_pidsley_all):,}")
print(f"• Zhou 2016 (MASK_*)            → in dataset: {len(S_zhou):,}    / total in list: {len(S_zhou_all):,}")
print(f"• Chen 2013 (cross-reactive)    → in dataset: {len(S_chenx):,}   / total in list: {len(S_chenx_all):,}")

# ---------------- union to remove + apply filter ----------------
removed_cpgs = S_naeem | S_pidsley | S_zhou | S_chenx

if layout == "rows":
    lf_filt = lf.filter(~pl.col(id_col).is_in(list(removed_cpgs)))
else:
    keep_cols = [c for c in schema_cols if c not in removed_cpgs]
    lf_filt = lf.select(keep_cols)

# artifacts
pl.DataFrame({"removed_cpg": sorted(removed_cpgs)}).write_csv("removed_cpgs.csv")

n_cols_after = len(lf_filt.schema)
n_rows_after = lf_filt.select(pl.len()).collect().item()

print("\nSummary:")
print(f"  • Samples × Probes (before): {n_rows_before} × {n_cols_before}")
print(f"  • Samples × Probes (after):  {n_rows_after} × {n_cols_after}")
print(f"  • Total removed CpGs:        {len(removed_cpgs):,}")
print(f"    - from Naeem:   {len(S_naeem):,}")
print(f"    - from Pidsley: {len(S_pidsley):,}")
print(f"    - from Zhou:    {len(S_zhou):,}")
print(f"    - from Chen-X:  {len(S_chenx):,}")

# ---------------- overlaps (exclusive partitions for 4 sets) ----------------
lists: Dict[str, Set[str]] = {
    "Naeem": S_naeem,
    "Pidsley": S_pidsley,
    "Zhou": S_zhou,
    "ChenX": S_chenx,
}
keys = list(lists.keys())

# compute exclusive counts for all non-empty combinations (2^4-1 = 15)
exclusive_counts: Dict[Tuple[str,...], int] = {}
U_all = set().union(*lists.values())
for r in range(1, len(keys)+1):
    for combo in combinations(keys, r):
        inter = set.intersection(*(lists[k] for k in combo))
        others = set().union(*(lists[k] for k in keys if k not in combo))
        excl = inter - others
        exclusive_counts[combo] = len(excl)

print("\nExclusive overlaps (present CpGs only):")
for r in range(1, len(keys)+1):
    for combo in combinations(keys, r):
        lbl = " ∩ ".join(combo)
        print(f"  • {lbl}: {exclusive_counts[combo]:,}")

# save detailed per-CpG source log (4 flags + source count)
pl.DataFrame(
    [{"CpG_ID": cg,
      "From_Naeem":   int(cg in S_naeem),
      "From_Pidsley": int(cg in S_pidsley),
      "From_Zhou":    int(cg in S_zhou),
      "From_ChenX":   int(cg in S_chenx)} for cg in removed_cpgs]
).with_columns(
    (pl.col("From_Naeem")+pl.col("From_Pidsley")+pl.col("From_Zhou")+pl.col("From_ChenX")).alias("Source_Count")
).sort(["Source_Count","CpG_ID"], descending=[True, False])\
 .write_csv("removed_cpgs_log.csv")

print("\nArtifacts:")
print("  • saved removed list → removed_cpgs.csv")
print("  • saved detailed log → removed_cpgs_log.csv")



Loading technical filter lists...
• Naeem 2014 (discard flag)     → in dataset: 190,672   / total in list: 190,672
• Pidsley/Chen (full blacklist) → in dataset: 39,737 / total in list: 67,030
• Zhou 2016 (MASK_*)            → in dataset: 96,541    / total in list: 193,182
• Chen 2013 (cross-reactive)    → in dataset: 29,233   / total in list: 29,233

Summary:
  • Samples × Probes (before): 407 × 485514
  • Samples × Probes (after):  407 × 260822
  • Total removed CpGs:        224,692
    - from Naeem:   190,672
    - from Pidsley: 39,737
    - from Zhou:    96,541
    - from Chen-X:  29,233

Exclusive overlaps (present CpGs only):
  • Naeem: 116,472
  • Pidsley: 1,004
  • Zhou: 21,845
  • ChenX: 556
  • Naeem ∩ Pidsley: 1,144
  • Naeem ∩ Zhou: 44,543
  • Naeem ∩ ChenX: 1,500
  • Pidsley ∩ Zhou: 1,958
  • Pidsley ∩ ChenX: 2,789
  • Zhou ∩ ChenX: 7
  • Naeem ∩ Pidsley ∩ Zhou: 8,493
  • Naeem ∩ Pidsley ∩ ChenX: 4,686
  • Naeem ∩ Zhou ∩ ChenX: 32
  • Pidsley ∩ Zhou ∩ ChenX: 5,861
  • Naeem

In [5]:
# --- Technical filters on Polars beta-matrix (now incl. Chen-2013 cross-reactive + McCartney-2016) ---
import polars as pl
from typing import Iterable, Set, Dict, Tuple
from itertools import combinations

# Paths
NAEEM_CSV        = "/kaggle/input/filtering-cpg-naeem/filtering_table.csv"                    # Naeem 2014
PIDSLEY_CSV      = "/kaggle/input/filtering-cpg-pidsley/Chen2013_full_blacklist.csv"          # Pidsley/Chen 2016/2013 (full blacklist)
CHEN_CROSS_CSV   = "/kaggle/input/filtering-cpg-chen-cross-reactive/chen_2013_cross_reactive.csv"  # Chen 2013 cross-reactive (first col)
ZHOU_TSV         = "/kaggle/input/filtering-cpg-zhou/EPIC.anno.GRCh38.tsv"                    # Zhou 2016 MASK_* annotation

# McCartney 2016 (Supplementary Table 2 & 3)
MCCARTNEY_CROSS_CPG    = "/kaggle/input/filtering-cpg-mccartney-2/1-s2.0-S221359601630071X-mmc2.txt"     # File 2
MCCARTNEY_CROSS_NONCG  = "/kaggle/input/filtering-cpg-mccartney-3/1-s2.0-S221359601630071X-mmc3.txt" # File 3

# ---------------- helpers to load lists ----------------
def load_naeem_drop_set(path: str,
                        probe_cands: Iterable[str]=("probe","IlmnID","ID_REF","cg","Name"),
                        flag_cands:  Iterable[str]=("Flag(discard/keep)","flag(discard/keep)","flag","flag_discard_keep")) -> Set[str]:
    """Return probes flagged as 'discard' in Naeem table."""
    df = pl.read_csv(path)
    probe_col = next((c for c in probe_cands if c in df.columns), None)
    flag_col  = next((c for c in df.columns if str(c).lower().strip().startswith("flag")), None) or next((c for c in flag_cands if c in df.columns), None)
    assert probe_col and flag_col, f"Naeem file columns not found. Seen: {df.columns}"
    probes = df[probe_col].cast(pl.Utf8).str.strip_chars()
    flags  = df[flag_col].cast(pl.Utf8).str.to_lowercase()
    return set(probes.filter(flags.str.contains("discard")).to_list())

def load_pidsley_drop_set(path: str, skip_first_lines: int = 1) -> Set[str]:
    """Robust loader for Pidsley/Chen full blacklist (CSV/TSV)."""
    df_comma = pl.read_csv(path, has_header=True, infer_schema_length=10000,
                           separator=",", skip_rows=skip_first_lines)
    try:
        df_tab = pl.read_csv(path, has_header=True, infer_schema_length=10000,
                             separator="\t", skip_rows=skip_first_lines)
    except Exception:
        df_tab = None
    cand = df_tab if (df_tab is not None and df_tab.width > df_comma.width) else df_comma

    cpg_like_cols = [c for c in cand.columns
                     if ("cg" in str(c).lower()) or ("ilmn" in str(c).lower())
                     or ("probe" in str(c).lower()) or ("id" in str(c).lower())]
    if not cpg_like_cols:
        cpg_like_cols = [cand.columns[0]]

    series_list = cand.select(pl.concat_list([pl.col(c).cast(pl.Utf8) for c in cpg_like_cols]).alias("_u")).to_series()
    s = series_list.explode().cast(pl.Utf8).str.strip_chars()
    values = s.to_list()
    return set(x for x in values if isinstance(x, str) and (x.startswith("cg") or x.startswith("ch")))

def load_firstcol_probe_list(path: str) -> Set[str]:
    """
    Robust 'first column' loader for simple lists (CSV/TSV, with or without header).
    Keeps ONLY the first column; returns cg*/ch* IDs.
    """
    def _try_read(sep: str, has_header: bool):
        return pl.read_csv(path, separator=sep, has_header=has_header, infer_schema_length=10000)

    df = None
    for sep in (",", "\t"):
        for hh in (True, False):
            try:
                df = _try_read(sep, hh)
                break
            except Exception:
                df = None
        if df is not None:
            break
    if df is None:  # last resort
        df = pl.read_csv(path, infer_schema_length=10000)

    first_col = df.columns[0]
    s = df[first_col].cast(pl.Utf8).str.strip_chars()
    vals = s.to_list()
    return set(x for x in vals if isinstance(x, str) and (x.startswith("cg") or x.startswith("ch")))

def load_chen_crossreactive_firstcol(path: str) -> Set[str]:
    """Chen 2013 cross-reactive list: CSV; keep ONLY the first column with probe IDs."""
    df = pl.read_csv(path, has_header=True, infer_schema_length=10000, separator=",")
    first_col = df.columns[0]
    s = df[first_col].cast(pl.Utf8).str.strip_chars()
    vals = s.to_list()
    return set(x for x in vals if isinstance(x, str) and (x.startswith("cg") or x.startswith("ch")))

def zhou_mask_ids(tsv_path: str,
                  id_col_candidates: Iterable[str] = ("probeID","IlmnID","Name")) -> Set[str]:
    """Read only ID + MASK_* columns from Zhou TSV/TSV.GZ and return probes to mask."""
    hdr = pl.read_csv(tsv_path, separator="\t", has_header=True, n_rows=1, ignore_errors=True)
    id_col = next((c for c in id_col_candidates if c in hdr.columns), None)
    assert id_col, f"ID column not found in Zhou TSV. Seen: {hdr.columns}"
    mask_cols = [c for c in hdr.columns if str(c).upper().startswith("MASK")]
    assert mask_cols, "No MASK_* columns found in Zhou TSV."

    df = pl.read_csv(tsv_path, separator="\t", has_header=True, columns=[id_col]+mask_cols,
                     dtypes={id_col: pl.Utf8}, ignore_errors=True)

    truthy = {"true","1","t","yes","y"}
    falsy  = {"false","0","f","no","n",""}
    def mask_to_bool(colname: str) -> pl.Expr:
        s = pl.col(colname).cast(pl.Utf8).str.to_lowercase()
        return (pl.when(s.is_in(list(truthy))).then(True)
                  .when(s.is_in(list(falsy))).then(False)
                  .otherwise(False)
                  .alias(colname))
    df_norm = df.select([pl.col(id_col).cast(pl.Utf8).str.strip_chars().alias(id_col)] + [mask_to_bool(c) for c in mask_cols])

    mask_any = pl.any_horizontal([pl.col(c) for c in mask_cols])
    bad = df_norm.filter(mask_any)[id_col].to_list()
    return set(bad)

# ---------------- dataset layout (LazyFrame/DataFrame) ----------------
lf = GSE69914.lazy() if isinstance(GSE69914, pl.DataFrame) else GSE69914
schema_cols = list(lf.schema.keys())
n_cols_before = len(schema_cols)
n_rows_before = lf.select(pl.len()).collect().item()

id_col_candidates = ("IlmnID","probeID","ID_REF","CpG","cg_id","Name")
id_col = next((c for c in id_col_candidates if c in schema_cols), None)

if id_col:
    present_cpgs = set(
        lf.select(pl.col(id_col).cast(pl.Utf8).str.strip_chars().alias("_id")).unique()
          .collect().get_column("_id").to_list()
    )
    layout = "rows"
else:
    cpg_cols = [c for c in schema_cols if isinstance(c, str) and (c.startswith("cg") or c.startswith("ch"))]
    present_cpgs = set(cpg_cols)
    layout = "cols"

# ---------------- load lists + intersect with present CpGs ----------------
print("Loading technical filter lists...")
S_naeem_all     = load_naeem_drop_set(NAEEM_CSV)
S_pidsley_all   = load_pidsley_drop_set(PIDSLEY_CSV)
S_zhou_all      = zhou_mask_ids(ZHOU_TSV)
S_chenx_all     = load_chen_crossreactive_firstcol(CHEN_CROSS_CSV)

# McCartney = union(File2 CpG-cross, File3 non-CpG-cross) -> treated as ONE source
S_mccartney2_all = load_firstcol_probe_list(MCCARTNEY_CROSS_CPG)
S_mccartney3_all = load_firstcol_probe_list(MCCARTNEY_CROSS_NONCG)
S_mccartney_all  = S_mccartney2_all | S_mccartney3_all

S_naeem     = S_naeem_all     & present_cpgs
S_pidsley   = S_pidsley_all   & present_cpgs
S_zhou      = S_zhou_all      & present_cpgs
S_chenx     = S_chenx_all     & present_cpgs
S_mccartney = S_mccartney_all & present_cpgs

print(f"• Naeem 2014 (discard flag)                    → in dataset: {len(S_naeem):,}     / total in list: {len(S_naeem_all):,}")
print(f"• Pidsley/Chen (full blacklist)                → in dataset: {len(S_pidsley):,}   / total in list: {len(S_pidsley_all):,}")
print(f"• Zhou 2016 (MASK_*)                           → in dataset: {len(S_zhou):,}      / total in list: {len(S_zhou_all):,}")
print(f"• Chen 2013 (cross-reactive)                   → in dataset: {len(S_chenx):,}     / total in list: {len(S_chenx_all):,}")
print(f"• McCartney 2016 (cross-hyb CpG + non-CpG)     → in dataset: {len(S_mccartney):,} / total in list: {len(S_mccartney_all):,}")

# ---------------- union to remove + apply filter ----------------
removed_cpgs = S_naeem | S_pidsley | S_zhou | S_chenx | S_mccartney

if layout == "rows":
    lf_filt = lf.filter(~pl.col(id_col).is_in(list(removed_cpgs)))
else:
    keep_cols = [c for c in schema_cols if c not in removed_cpgs]
    lf_filt = lf.select(keep_cols)

# artifacts
pl.DataFrame({"removed_cpg": sorted(removed_cpgs)}).write_csv("removed_cpgs.csv")

n_cols_after = len(lf_filt.schema)
n_rows_after = lf_filt.select(pl.len()).collect().item()

print("\nSummary:")
print(f"  • Samples × Probes (before): {n_rows_before} × {n_cols_before}")
print(f"  • Samples × Probes (after):  {n_rows_after} × {n_cols_after}")
print(f"  • Total removed CpGs:        {len(removed_cpgs):,}")
print(f"    - from Naeem:      {len(S_naeem):,}")
print(f"    - from Pidsley:    {len(S_pidsley):,}")
print(f"    - from Zhou:       {len(S_zhou):,}")
print(f"    - from Chen-X:     {len(S_chenx):,}")
print(f"    - from McCartney:  {len(S_mccartney):,}")

# ---------------- overlaps (exclusive partitions; dynamic for 5 lists = 31 combos) ----------------
lists: Dict[str, Set[str]] = {
    "Naeem": S_naeem,
    "Pidsley": S_pidsley,
    "Zhou": S_zhou,
    "ChenX": S_chenx,
    "McCartney": S_mccartney,
}
keys = list(lists.keys())

exclusive_counts: Dict[Tuple[str,...], int] = {}
for r in range(1, len(keys)+1):
    for combo in combinations(keys, r):
        inter = set.intersection(*(lists[k] for k in combo))
        others = set().union(*(lists[k] for k in keys if k not in combo))
        excl = inter - others
        exclusive_counts[combo] = len(excl)

print("\nExclusive overlaps (present CpGs only):")
for r in range(1, len(keys)+1):
    for combo in combinations(keys, r):
        lbl = " ∩ ".join(combo)
        print(f"  • {lbl}: {exclusive_counts[combo]:,}")

# save detailed per-CpG source log (5 flags + source count)
pl.DataFrame(
    [{"CpG_ID": cg,
      "From_Naeem":     int(cg in S_naeem),
      "From_Pidsley":   int(cg in S_pidsley),
      "From_Zhou":      int(cg in S_zhou),
      "From_ChenX":     int(cg in S_chenx),
      "From_McCartney": int(cg in S_mccartney)} for cg in removed_cpgs]
).with_columns(
    (pl.col("From_Naeem")+pl.col("From_Pidsley")+pl.col("From_Zhou")+pl.col("From_ChenX")+pl.col("From_McCartney")).alias("Source_Count")
).sort(["Source_Count","CpG_ID"], descending=[True, False])\
 .write_csv("removed_cpgs_log.csv")

print("\nArtifacts:")
print("  • saved removed list → removed_cpgs.csv")
print("  • saved detailed log → removed_cpgs_log.csv")

# Optionally persist the filtered dataset:
# GSE69914_filt = lf_filt
# GSE69914_filt.collect().write_parquet("GSE69914_filtered.parquet")


Loading technical filter lists...
• Naeem 2014 (discard flag)                    → in dataset: 190,672     / total in list: 190,672
• Pidsley/Chen (full blacklist)                → in dataset: 39,737   / total in list: 67,030
• Zhou 2016 (MASK_*)                           → in dataset: 96,541      / total in list: 193,182
• Chen 2013 (cross-reactive)                   → in dataset: 29,233     / total in list: 29,233
• McCartney 2016 (cross-hyb CpG + non-CpG)     → in dataset: 28,582 / total in list: 44,208

Summary:
  • Samples × Probes (before): 407 × 485514
  • Samples × Probes (after):  407 × 260088
  • Total removed CpGs:        225,426
    - from Naeem:      190,672
    - from Pidsley:    39,737
    - from Zhou:       96,541
    - from Chen-X:     29,233
    - from McCartney:  28,582

Exclusive overlaps (present CpGs only):
  • Naeem: 116,068
  • Pidsley: 987
  • Zhou: 21,787
  • ChenX: 556
  • McCartney: 734
  • Naeem ∩ Pidsley: 1,063
  • Naeem ∩ Zhou: 44,424
  • Naeem ∩ ChenX: 1

734